# MF1 Nubar & Fission Energy Parsing

Verification notebook for MT452 (total nubar), MT455 (delayed nubar),
MT456 (prompt nubar), and MT458 (fission energy release) parsing
using U-235 JEFF-4.0.

In [ ]:
from kika.endf.read_endf import read_endf
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
filepath = '../../../files/endf/U235_jeff4.0_n.endf'
endf = read_endf(filepath, mf_numbers=[1])
mf1 = endf.mf[1]
print('MF1 sections:', list(mf1.sections.keys()))

## MT452 — Total Nubar

In [ ]:
mt452 = mf1.get_section(452)
print(repr(mt452))
print(f'Representation: {mt452.representation}')
print(f'LNU: {mt452.lnu}')
print(f'Number of energy points: {len(mt452.energies)}')
print(f'Energy range: {mt452.energies[0]:.2e} — {mt452.energies[-1]:.2e} eV')
print(f'Nubar range: {min(mt452.nubar_values):.4f} — {max(mt452.nubar_values):.4f}')

## MT456 — Prompt Nubar

In [ ]:
mt456 = mf1.get_section(456)
print(repr(mt456))
print(f'Representation: {mt456.representation}')
print(f'Number of energy points: {len(mt456.energies)}')
print(f'Energy range: {mt456.energies[0]:.2e} — {mt456.energies[-1]:.2e} eV')
print(f'Nubar range: {min(mt456.nubar_values):.4f} — {max(mt456.nubar_values):.4f}')

## MT455 — Delayed Nubar

In [ ]:
mt455 = mf1.get_section(455)
print(repr(mt455))
print(f'LDG: {mt455.ldg} ({"energy-independent" if mt455.ldg == 0 else "energy-dependent"} decay)')
print(f'LNU: {mt455.lnu} ({mt455.representation})')
print(f'Precursor families: {mt455.num_precursor_families}')
print(f'Decay constants (1/s): {mt455.decay_constants}')
print(f'Number of nubar points: {len(mt455.energies)}')
print(f'Nubar range: {min(mt455.nubar_values):.6f} — {max(mt455.nubar_values):.6f}')

## MT458 — Fission Energy Release

In [ ]:
mt458 = mf1.get_section(458)
print(repr(mt458))
print(f'LFC: {mt458.lfc}, Polynomial order: {mt458.polynomial_order}')
print()

components = [
    ('EFR', 'Fission fragment KE', mt458.efr),
    ('ENP', 'Prompt neutron KE', mt458.enp),
    ('END', 'Delayed neutron KE', mt458.end),
    ('EGP', 'Prompt gamma energy', mt458.egp),
    ('EGD', 'Delayed gamma energy', mt458.egd),
    ('EB',  'Delayed beta energy', mt458.eb),
    ('ENU', 'Neutrino energy', mt458.enu),
    ('ER',  'Total less neutrinos', mt458.er),
    ('ET',  'Total fission energy', mt458.et),
]

print(f'{"Component":<6} {"Description":<25} {"Value (eV)":>15} {"Uncertainty":>15}')
print('-' * 65)
for name, desc, (val, unc) in components:
    print(f'{name:<6} {desc:<25} {val:>15.1f} {unc:>15.1f}')

## Round-Trip Verification

In [ ]:
# Read original file lines grouped by MT
with open(filepath, 'r') as f:
    all_lines = f.readlines()

original_mt_lines = {}
for line in all_lines:
    if len(line) >= 75:
        try:
            mf_n = int(line[70:72].strip())
            mt_n = int(line[72:75].strip())
            if mf_n == 1 and mt_n > 0:
                if mt_n not in original_mt_lines:
                    original_mt_lines[mt_n] = []
                original_mt_lines[mt_n].append(line.rstrip())
        except:
            pass

# Test round-trip for each section
for mt_num in [452, 455, 456, 458]:
    sec = mf1.get_section(mt_num)
    serialized = str(sec)
    ser_lines = serialized.split('\n')
    # Remove SEND line from serialized output
    data_lines = [l for l in ser_lines if not (len(l) >= 75 and l[72:75].strip() == '0')]
    orig_lines = original_mt_lines.get(mt_num, [])

    if len(data_lines) != len(orig_lines):
        print(f'MT{mt_num}: FAIL (line count: {len(data_lines)} vs {len(orig_lines)})')
        continue

    mismatches = sum(1 for s, o in zip(data_lines, orig_lines) if s[:66] != o[:66])
    if mismatches == 0:
        print(f'MT{mt_num}: PASS ({len(data_lines)} lines match)')
    else:
        print(f'MT{mt_num}: FAIL ({mismatches} mismatches in {len(data_lines)} lines)')

## Nubar Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Total nubar
ax.plot(mt452.energies, mt452.nubar_values, label='Total ($\\bar{\\nu}$)', color='black', linewidth=1.5)

# Prompt nubar
ax.plot(mt456.energies, mt456.nubar_values, label='Prompt ($\\bar{\\nu}_p$)', color='blue', linewidth=1.0, linestyle='--')

# Delayed nubar
ax.plot(mt455.energies, mt455.nubar_values, label='Delayed ($\\bar{\\nu}_d$)', color='red', linewidth=1.0, linestyle='-.')

ax.set_xscale('log')
ax.set_xlabel('Energy (eV)')
ax.set_ylabel('$\\bar{\\nu}$ (neutrons/fission)')
ax.set_title('U-235 JEFF-4.0 — Nubar')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Consistency Check: nu_total = nu_prompt + nu_delayed

In [ ]:
# Compare at the delayed nubar energy points (coarser grid)
from kika.endf.utils import interpolate_1d_endf

e_delayed = np.array(mt455.energies)
nu_d = np.array(mt455.nubar_values)

# Interpolate total and prompt at delayed nubar energies
nu_total_at_ed = interpolate_1d_endf(
    mt452.energies, mt452.nubar_values, mt452.interpolation, e_delayed
)
nu_prompt_at_ed = interpolate_1d_endf(
    mt456.energies, mt456.nubar_values, mt456.interpolation, e_delayed
)

residual = nu_total_at_ed - (nu_prompt_at_ed + nu_d)
print('Consistency check: nu_total - (nu_prompt + nu_delayed)')
print(f'Max absolute residual: {np.max(np.abs(residual)):.2e}')
print(f'Mean absolute residual: {np.mean(np.abs(residual)):.2e}')
print()
for i, e in enumerate(e_delayed):
    print(f'E={e:12.4e}  total={nu_total_at_ed[i]:.6f}  prompt={nu_prompt_at_ed[i]:.6f}  delayed={nu_d[i]:.6f}  residual={residual[i]:.2e}')